# 1 — Create `df_log` from the log files

**This notebook makes the dataset. It does not analyse it** — that is notebook 2.

Everything here comes from `log.json` and nothing else: no video, no optic flow. That is what lets
it run over a whole server directory whose sessions were never put through the video pipeline.

It produces exactly **two tables**:

| | one row per | holds |
|---|---|---|
| `df_sessions` | SESSION | who/when/which world/which protocol + **the performance numbers (D, chance, throughput…)** |
| `df_trials` | TRIAL | the trial window, path geometry, the on-screen icons, the outcome, the cluster and the error/conflict labels |

Both are saved to `MAIN_DIR/df_log/`. Notebook 2 loads them and never touches the server again.

**The trial logic is not written here.** `task_*/build_trials.py` already defines a trial correctly —
a **spawn batch**, so a batch that ends without a collection (a reshuffle, the session-end tail) is a
real row — and it carries the `icons` list, the `NORMAL`/`BANISH_WORLD` column, and
`start_frame`/`end_frame`. Those builders are called here directly, and **nothing is read from or
written into the session folders**.

Sections **A** are sanity checks on the data; sections **B** build; section **C** verifies and saves.

## Config ← YOU SET THIS

In [ ]:
MAIN_DIR = '/mnt/server/data'     # the directory holding one folder per animal
VIEW_SCALES = {}                  # {world signature: scale} for any world without a known scale
PIPELINE_DIR = None               # None = locate session_pipeline/ automatically
# =============================================================================

import sys, json, importlib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

MAIN_DIR = Path(MAIN_DIR).expanduser()
cands = ([Path(PIPELINE_DIR)] if PIPELINE_DIR else []) + [
    Path.cwd().parent, Path.cwd(), Path.cwd().parent / 'session_pipeline']
PIPE = next((c.resolve() for c in cands if (c / 'common' / 'session_index.py').exists()), None)
if PIPE is None:
    raise FileNotFoundError('set PIPELINE_DIR. Tried: ' + ', '.join(str(c) for c in cands))
sys.path.insert(0, str(PIPE / 'common')); sys.path.insert(0, str(PIPE / 'performance'))

import session_index as sidx, perf_from_log as pfl, build_log_df as bl
# RELOAD, don't just import: Python caches a module, so after a `git pull` the kernel would keep
# running the OLD code while re-running this cell looked like it worked.
sidx = importlib.reload(sidx); pfl = importlib.reload(pfl); bl = importlib.reload(bl)

print(f'pipeline : {PIPE}')
print(f'main dir : {MAIN_DIR}')
print(f'builders : {list(bl.TASK_MODULES)}   (others fall back to the generic builder)')

## A1 — SANITY: what is actually on disk?

Deliberately dumb: it lists folders and counts files, **without opening a single log**. If the mount
is missing, the path is mis-typed, or a sync is half-finished, it looks wrong *here* — rather than
showing up later as a mysteriously small dataset.

In [ ]:
ANIMALS = sidx.list_animals(MAIN_DIR)
display(ANIMALS)

for a in ANIMALS.animal:
    print(f'\n--- {a} ---')
    display(sidx.list_sessions(MAIN_DIR / a).head(8))

## A2 — What a log actually contains

One log, opened and shown, so the rest of the notebook is readable: you can see the fields every
column below is derived from.

In [ ]:
_first = next((MAIN_DIR / a).glob('*/log.json'), None) or next(MAIN_DIR.glob('*/*/log.json'))
_L = json.load(open(_first))
print(_first, '\n')
print('top-level keys :', list(_L.keys()), '\n')
print('experiment_data:', _L.get('experiment_data'), '\n')
# collected[]: one row per collection. It carries the reward `multiplier` (for the current world and
# the next), the effect taken, and where.
print('collected[0]   :', (_L.get('collected') or [{}])[0], '\n')
# spawns[]: each has `current` = the icons ON THE BOARD at that spawn. THIS is what the task is read
# from, per trial -- shown in full for the first spawn so the source is visible, not hidden.
print('spawns[0]      :', (_L.get('spawns') or [{}])[0], '\n')
print('worlds[0]      :', (_L.get('worlds') or [{}])[0], '\n')

# where the TASK comes from: the board (current icons) of each of the first trials -> classified task.
# Line this up against the video to confirm it is reading the right thing.
pfl.show_first_trials(_first)

## A3 — SANITY: read every log and check the identities

`discover()` opens each log and reads what it refuses to guess: the **date**
(`experiment_data.datetime`), the **animal** (the ID's digits, falling back to the folder name), and
the **task**. The task is decided from the **first 10 trials' boards** (`pfl.first_trials_task`): each
trial's task is read from that spawn batch's icons, and if the first 10 are one task the session is
that task. A single differing trial-0 board is allowed (the timeout→banishment shaping switch);
anything less consistent is left with its settled guess but flagged `task_stable = False` so it is
**shown** rather than trusted. The whole-log union of effects is kept as `task_union` for comparison.

Everything printed here is a **check**, not a result.

In [ ]:
S = pd.concat([sidx.discover(MAIN_DIR / a, view_scales=VIEW_SCALES) for a in ANIMALS.animal],
               ignore_index=True)
print(f'\n{len(S)} session(s), {S.mouse.nunique()} animal(s): {sorted(S.mouse.dropna().unique())}')

# animal resolved from the FOLDER rather than the log ID?
if (S.mouse_src == 'folder').any():
    n = int((S.mouse_src == 'folder').sum())
    print(f'\n{n} session(s) took the animal from the folder name (the log ID had no number):')
    display(S.loc[S.mouse_src == 'folder', ['session', 'mouse_raw', 'mouse']].head())

# days holding more than one session
dup = S.day.duplicated(keep=False)
if dup.any():
    print('\ndays with more than one session:')
    display(S.loc[dup, ['session', 'day', 'time', 'name']])

display(S[['session', 'mouse', 'day', 'task', 'task_stable', 'world', 'view_scale', 'use', 'note']].head(12))

### A4 — how many sessions of each task?

The number this notebook is for: how many sessions of each protocol, and specifically
**banish_multiplier vs timeout_multiplier**. The task is the **majority of each session's first 10
trials** (a clean trial-0 → banishment switch still counts as the task it settles into). A session
with no clear majority is listed as `task_stable = False` so you can check it against the video with
`pfl.show_first_trials(log_path)`.

(World numbering is deliberately not shown here — the log has no world id, and the invented W-labels
don't match the lab's region-based worlds. `sidx.task_protocol(S)` still prints them if ever needed.)

In [ ]:
print('TASKS in this directory:')
print(S.task.value_counts().to_string())

print(f"\nbanish_multiplier : {int((S.task == 'banish_multiplier').sum())}")
print(f"timeout_multiplier: {int((S.task == 'timeout_multiplier').sum())}")

# sessions with no clear task majority in their first 10 trials -> check against the video
_uns = S[~S.task_stable.fillna(False)]
if len(_uns):
    print(f'\n{len(_uns)} session(s) with no clear task majority in the first 10 trials:')
    display(_uns[['session', 'task', 'task_seq']])
else:
    print('\nevery session has a clear task majority in its first 10 trials.')

## B1 — BUILD the two tables

One call. For each session: the task's `build_trials` → `cluster_paths` → `label_trials` (all
log-only), then the performance numbers merged into the session row.

**Every discovered session becomes a row.** One that cannot be scored keeps its identity, gets `NaN`
performance and a reason in `perf_error`, and stays in the table — filtering is notebook 2's
explicit choice, not a deletion baked in here. Days holding two sessions are **flagged**
(`is_dup_day`, `keep_of_day`), not dropped.

In [ ]:
df_sessions, df_trials = bl.build_all(MAIN_DIR, view_scales=VIEW_SCALES)

## Column glossary (the ones that confuse)

**Trial outcome counts** — `n_single_reward`, `n_banish`, `n_unbanish`, `n_timeout`, … one per outcome. Two are not collections:
- **`n_reshuffle`** — trials where the GAME respawned the **whole board** mid-session with **no collection** (the icons were reshuffled). It is the game reshuffling, not the mouse doing anything; these are excluded from analysis (`analyze = False`).
- **`n_incomplete`** — the **session-end tail**: the final spawn batch that never ended in a collection because the recording stopped. Also excluded.

**Protocol switch — the BOARD changing task, NOT the camera:**
- **`switch_ms`** — if the session OPENED under one task and switched to another at trial 0 (e.g. timeout → banishment), the time of that switch; `NaN` if it never switched.
- **`n_before_switch`** — how many trials came before the switch. Performance is scored only on the part AFTER the switch, so those trials are excluded from the score (they still appear in `df_trials`).

**Camera check — only present after the video/tracking pipeline has run:**
- **`camera_stable`** — `True` = checked and steady · `False` = camera MOVED mid-session · `None` = not checked yet (tracking hasn't run on this session).
- **`camera_move_frame` / `camera_move_ms`** — where the shift began (`NaN` if stable or unchecked).

## C1 — VERIFY the dataset

Checks that it is right, not that it is interesting. Anything printing `FAIL` needs looking at
before the tables are used.

In [ ]:
def check(name, ok, detail=''):
    print(f'  [{"PASS" if ok else "FAIL"}] {name}' + (f'   {detail}' if detail else ''))

print('DATASET CHECKS')
check('every session has a unique name', df_sessions.session.is_unique)
check('every trial belongs to a listed session',
      set(df_trials.session) <= set(df_sessions.session))

# trial counts agree between the two tables
per = df_trials.groupby('session').size()
agree = all(int(per.get(r.session, 0)) == int(r.n_trials_total) for _, r in df_sessions.iterrows())
check('trials per session match df_sessions.n_trials_total', agree)

# reward drops, computed two independent ways. score_log scores only the part AFTER a protocol
# switch (after_switch=True), so for a switch session its SESSION total excludes the trial-0
# lead-in while the TRIAL table keeps every trial. Show BOTH the all-trials sum and the post-switch
# sum, and pass if EITHER matches the session total -- so a genuine mismatch stands out from the
# benign switch-boundary case, which is labelled by switch_ms / n_before_switch.
_sw = df_sessions.set_index('session').switch_ms if 'switch_ms' in df_sessions else pd.Series(dtype=float)
_g = df_trials.groupby('session')
def _sums(s):
    m = df_trials[df_trials.session == s]
    allsum = float(m.drops.sum())
    post = allsum
    if s in _sw.index and pd.notna(_sw.get(s)) and 'start_ms' in m:
        post = float(m[m.start_ms >= _sw[s]].drops.sum())
    return allsum, post
_d_se = df_sessions.set_index('session').drops
_both = [s for s in _d_se.index if pd.notna(_d_se[s]) and s in set(df_trials.session)]
_rows = []
for s in _both:
    a, po = _sums(s)
    tot = float(_d_se[s])
    if min(abs(a - tot), abs(po - tot)) > 1e-6:      # neither interpretation matches -> real
        _rows.append((s, a, po, tot))
check('reward drops agree (trial sum vs session total)', not _rows, f'{len(_both)} session(s)')
if _rows:
    print(f'    {len(_rows)} session(s) disagree -- neither all-trials nor post-switch sum matches:')
    _bt = pd.DataFrame(_rows, columns=['session', 'trial_sum_all', 'trial_sum_postswitch',
                                       'session_total'])
    _cols = [c for c in ['session', 'task', 'switch_ms', 'n_before_switch'] if c in df_sessions]
    display(_bt.merge(df_sessions[_cols], on='session', how='left').head(20))

check('no session is missing a world', df_sessions.world_sig.notna().all())
n_bad = int((df_sessions.perf_error != '').sum())
check('all sessions scored', n_bad == 0, f'{n_bad} without performance')
if n_bad:
    display(df_sessions.loc[df_sessions.perf_error != '', ['session', 'task', 'perf_error']])

gen = df_trials.builder.eq('generic').sum() if 'builder' in df_trials else 0
if gen:
    print(f'\n  NOTE {gen} trial(s) came from the GENERIC builder (no dedicated one for that '
          f'protocol yet): {sorted(df_trials.loc[df_trials.builder == "generic", "task"].unique())}')
    print('       They have the common columns only -- no spawn-batch trials, no cluster/labels.')

### C2 — Cross-check against a session built by the full pipeline

**This cross-check only makes sense AFTER the tracking (video) pipeline has run on a session.** It
compares the log-only trial table against the `df_trials_clean.pkl` that tracking produces, so it
needs that reference file to exist. On raw server data (log only) there is nothing to compare yet,
and the cell says so rather than doing it silently — that is expected, not a failure.

In [ ]:
_n_checked = 0
for _, r in df_sessions.iterrows():
    ref_p = Path(r['dir']) / 'df_trials_clean.pkl'
    if not ref_p.exists():
        continue
    _n_checked += 1
    mine = df_trials[df_trials.session == r.session]
    # Only compare where BOTH tables were built by the same task builder. A generic-builder session
    # defines a trial as collection-to-collection while the reference may be a spawn-batch table
    # (or, for the legacy JPAS_0231 file, an older overlapping-window schema entirely) -- comparing
    # those reports a difference of DEFINITION as though it were an error.
    if 'builder' in mine and (mine.builder == 'generic').any():
        print(f'{r.session}:  SKIPPED -- built by the generic builder, so the reference table is '
              f'a different trial definition, not a comparable one')
        continue
    ref = pd.read_pickle(ref_p)
    print(f'{r.session}:  mine {len(mine)} trials vs reference {len(ref)}')
    if len(mine) != len(ref):
        check('   same number of trials', False)
        continue
    check('   outcomes identical', (mine.outcome.values == ref.outcome.values).all())
    cols = [c for c in ['dur_s', 'path_efficiency', 'time_in_corner', 'mean_speed',
                        'heading_align'] if c in mine and c in ref]
    check('   geometry identical',
          all(np.allclose(mine[c].values.astype(float), ref[c].values.astype(float),
                          equal_nan=True) for c in cols), f'({", ".join(cols)})')
    if 'cluster_name' in ref and 'cluster_name' in mine:
        check('   cluster identical', (mine.cluster_name.values == ref.cluster_name.values).all())

if _n_checked == 0:
    print('No session in this directory has a df_trials_clean.pkl (none has been through the video '
          'pipeline), so there is nothing to cross-check here. This is expected for raw server data; '
          'the cross-check runs only where a full-pipeline reference table exists (e.g. flow_test).')

## C3 — SAVE

`.pkl` keeps everything, including the per-trial coordinate arrays and the `icons` lists. The
`.csv` of `df_sessions` is for eyeballing and sharing (it cannot carry the build stamp).

Each file is stamped with when and by what version it was built, so a stale copy is recognisable
rather than merely old.

In [ ]:
from pathlib import Path

# 1) canonical copy, next to the server data
OUT = MAIN_DIR / 'df_log'
OUT.mkdir(exist_ok=True)
for _f, _obj in [('df_sessions.pkl', df_sessions), ('df_trials.pkl', df_trials),
                 ('df_sessions.csv', df_sessions)]:
    bl.save(_obj, OUT / _f)

# 2) LOCAL copy, so notebook 2 (and you) can read it without the server mounted / re-downloading
LOCAL_OUT = Path('~/repo/session_pipeline_output').expanduser()
LOCAL_OUT.mkdir(parents=True, exist_ok=True)
for _f, _obj in [('df_sessions.pkl', df_sessions), ('df_trials.pkl', df_trials),
                 ('df_sessions.csv', df_sessions)]:
    bl.save(_obj, LOCAL_OUT / _f)

print(f'\nsaved to:\n  {OUT}\n  {LOCAL_OUT}   <- local copy, notebook 2 reads this one')
df_sessions.head()